In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Cities 2022 

In [6]:
wci_2022 = pd.read_csv("../data/city_safety/world-crime-index.csv")

In [7]:
wci_2022.head()

,Rank,City,Crime Index,Safety Index
0,1,"Caracas, Venezuela",83.98,16.02
1,2,"Pretoria, South Africa",81.98,18.02
2,3,"Celaya, Mexico",81.80,18.20
3,4,"San Pedro Sula, Honduras",80.87,19.13
4,5,"Port Moresby, Papua New Guinea",80.71,19.29


In [8]:
# Normalize column names (lowercase, underscores)
wci_2022.columns = (
    wci_2022.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [9]:
wci_2022.head()

,rank,city,crime_index,safety_index
0,1,"Caracas, Venezuela",83.98,16.02
1,2,"Pretoria, South Africa",81.98,18.02
2,3,"Celaya, Mexico",81.80,18.20
3,4,"San Pedro Sula, Honduras",80.87,19.13
4,5,"Port Moresby, Papua New Guinea",80.71,19.29


In [10]:
# Split city into city + country on comma
city_country = wci_2022['city'].str.rsplit(',', n=1, expand=True)

wci_2022['city'] = city_country[0].str.strip()
wci_2022['country'] = city_country[1].str.strip()

In [11]:
wci_2022.head()

,rank,city,crime_index,safety_index,country
0,1,Caracas,83.98,16.02,Venezuela
1,2,Pretoria,81.98,18.02,South Africa
2,3,Celaya,81.80,18.20,Mexico
3,4,San Pedro Sula,80.87,19.13,Honduras
4,5,Port Moresby,80.71,19.29,Papua New Guinea


In [12]:
# Add normalized versions for joins later
wci_2022['city_norm'] = wci_2022['city'].str.strip().str.lower()
wci_2022['country_norm'] = wci_2022['country'].str.strip().str.lower()

In [13]:
# Check for missing values
wci_2022.isna().sum()

rank            0
city            0
crime_index     0
safety_index    0
country         0
city_norm       0
country_norm    0
dtype: int64

In [14]:
wci_2022.isna().mean()

rank            0.0
city            0.0
crime_index     0.0
safety_index    0.0
country         0.0
city_norm       0.0
country_norm    0.0
dtype: float64

In [15]:
# Save cleaned dataframe back into data/city_safety
out_path = "../data/city_safety/world_crime_index_2022_clean.csv"
wci_2022.to_csv(out_path, index=False)

In [16]:
pd.read_csv("../data/city_safety/world_crime_index_2022_clean.csv").head()

,rank,city,crime_index,safety_index,country,city_norm,country_norm
0,1,Caracas,83.98,16.02,Venezuela,caracas,venezuela
1,2,Pretoria,81.98,18.02,South Africa,pretoria,south africa
2,3,Celaya,81.80,18.20,Mexico,celaya,mexico
3,4,San Pedro Sula,80.87,19.13,Honduras,san pedro sula,honduras
4,5,Port Moresby,80.71,19.29,Papua New Guinea,port moresby,papua new guinea


# Countries 2024

In [17]:
# Load global country-level crime/safety data
global_path = "../data/crime-rate-by-country-2024.csv"  # from https://www.kaggle.com/datasets/shahriarkabir/crime-rate-by-country-2024?resource=download
global_df = pd.read_csv(global_path)

global_df.head()

,country,crimeRateByCountry_crimeIndex,CrimeRate_OverallCriminalityScoreGOCI,CrimeRate_CriminalMarketsScore,CrimeRate_CriminalActorsScore,CrimeRate_ResilienceScore,CrimeRateSafetyIndex
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [18]:
print(global_df.columns)

Index(['country', 'crimeRateByCountry_crimeIndex',
       'CrimeRate_OverallCriminalityScoreGOCI',
       'CrimeRate_CriminalMarketsScore', 'CrimeRate_CriminalActorsScore',
       'CrimeRate_ResilienceScore', 'CrimeRateSafetyIndex'],
      dtype='object')


In [19]:
# Standardize column names
global_df.columns = (
    global_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Rename to a cleaner schema

global_df = global_df.rename(columns={
    "country_name": "country",
    "crimeratebycountry_crimeindex": "crime_index",
    "crimerate_overallcriminalityscoregoci": "overall_criminality_score",
    "crimerate_criminalmarketsscore": "criminal_markets_score",
    "crimerate_criminalactorsscore": "criminal_actors_score",
    "crimerate_resiliencescore": "resilience_score",
    "crimeratesafetyindex": "safety_index"
})

global_df.head()


,country,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [20]:
# Missing counts per column
print(global_df.isna().sum())

# Fraction missing per column
print(global_df.isna().mean())

# Which countries have any missing numeric values
num_cols = [
    'crime_index',
    'overall_criminality_score',
    'criminal_markets_score',
    'criminal_actors_score',
    'resilience_score',
    'safety_index',
]

global_df['num_missing'] = global_df[num_cols].isna().sum(axis=1)

countries_with_missing = global_df[global_df['num_missing'] > 0][
    ['country', 'num_missing'] + num_cols
]

countries_with_missing.head(50)

country                       0
crime_index                  57
overall_criminality_score     6
criminal_markets_score        6
criminal_actors_score         6
resilience_score              6
safety_index                 58
dtype: int64
country                      0.000000
crime_index                  0.287879
overall_criminality_score    0.030303
criminal_markets_score       0.030303
criminal_actors_score        0.030303
resilience_score             0.030303
safety_index                 0.292929
dtype: float64


,country,num_missing,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index
14,DR Congo,2,NaN,7.35,6.20,8.5,2.38,NaN
19,Thailand,2,NaN,6.18,6.77,5.6,4.79,NaN
27,Colombia,2,NaN,7.75,7.30,8.2,5.63,NaN
36,Yemen,2,NaN,6.57,5.63,7.5,1.75,NaN
37,Canada,2,NaN,3.88,3.87,3.9,7.21,NaN
48,Madagascar,2,NaN,5.58,5.27,5.9,3.33,NaN
51,Cameroon,2,NaN,6.27,6.23,6.3,3.17,NaN
53,Niger,2,NaN,5.70,5.70,5.7,3.46,NaN
57,Mali,2,NaN,5.93,6.47,5.4,2.38,NaN
59,Taiwan,4,16.1,NaN,NaN,NaN,NaN,83.9


In [ ]:
#out_path = "../data/country_safety/country_crime_safety_2024_clean.csv"
#global_df.to_csv(out_path, index=False)

# need to clean or decide on fill values before saving  

# Countries 2020 - this dataset came along with many other csvs that could be used as features

In [22]:
#https://www.kaggle.com/datasets/dumbgeek/countries-dataset-2020?select=Crime+index+by+countries+2020.csv
global_df_1 = pd.read_csv("../data/Crime_index_by_countries_2020.csv")
global_df_1.head()

,Country,Crime Index,Safety Index
0,Venezuela,84.49,15.51
1,Papua New Guinea,81.93,18.07
2,South Africa,77.49,22.51
3,Afghanistan,76.23,23.77
4,Honduras,76.11,23.89


In [23]:
print(global_df_1.columns)

Index(['Country', 'Crime Index', 'Safety Index'], dtype='object')


In [24]:
# Normalize names
global_df_1.columns = (
    global_df_1.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

global_df_1.head()

,country,crime_index,safety_index
0,Venezuela,84.49,15.51
1,Papua New Guinea,81.93,18.07
2,South Africa,77.49,22.51
3,Afghanistan,76.23,23.77
4,Honduras,76.11,23.89


In [25]:
# Clean country text and add normalized key for merges
global_df_1['country'] = global_df_1['country'].astype(str).str.strip()
global_df_1['country_norm'] = global_df_1['country'].str.lower()

# Ensure numeric dtypes (coerce any weird strings to NaN)
num_cols = ['crime_index', 'safety_index']

for c in num_cols:
    global_df_1[c] = pd.to_numeric(global_df_1[c], errors='coerce')

In [26]:
# Duplicated countries (after normalization)
dupes = global_df_1[global_df_1['country_norm'].duplicated(keep=False)]\
        .sort_values('country_norm')
print(dupes.head(20))

# Missing counts
print(global_df_1[num_cols].isna().sum())
print(global_df_1[num_cols].isna().mean())

Empty DataFrame
Columns: [country, crime_index, safety_index, country_norm]
Index: []
crime_index     0
safety_index    0
dtype: int64
crime_index     0.0
safety_index    0.0
dtype: float64


In [28]:
global_df_1 = global_df_1.rename(columns={
    'crime_index': 'crime_index_2020',
    'safety_index': 'safety_index_2020'
})

out_path = "../data/country_safety/country_crime_safety_2020_clean.csv"
global_df_1.to_csv(out_path, index=False)

In [38]:
pd.read_csv("../data/country_safety/country_crime_safety_2020_clean.csv").head()

,country,crime_index_2020,safety_index_2020,country_norm
0,Venezuela,84.49,15.51,venezuela
1,Papua New Guinea,81.93,18.07,papua new guinea
2,South Africa,77.49,22.51,south africa
3,Afghanistan,76.23,23.77,afghanistan
4,Honduras,76.11,23.89,honduras


# Countries 2023 

In [31]:
#https://www.kaggle.com/datasets/zabihullah18/crime-rate-by-country-2023
global_df_2 = pd.read_csv("../data/crime_index_by_country_2023.csv")
global_df_2.head()

,rank,country,crimeIndex,pop2023
0,1,Venezuela,83.76,28838499.0
1,2,Papua New Guinea,80.79,10329931.0
2,3,South Africa,76.86,60414495.0
3,4,Afghanistan,76.31,42239854.0
4,5,Honduras,74.54,10593798.0


In [32]:
global_df_2.columns = (
    global_df_2.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

global_df_2.head()

,rank,country,crimeindex,pop2023
0,1,Venezuela,83.76,28838499.0
1,2,Papua New Guinea,80.79,10329931.0
2,3,South Africa,76.86,60414495.0
3,4,Afghanistan,76.31,42239854.0
4,5,Honduras,74.54,10593798.0


In [33]:
global_df_2['country'] = global_df_2['country'].astype(str).str.strip()
global_df_2['country_norm'] = global_df_2['country'].str.lower()

In [34]:
num_cols = ['crimeindex', 'pop2023']

for c in num_cols:
    global_df_2[c] = pd.to_numeric(global_df_2[c], errors='coerce')

In [35]:
# Duplicated countries
dupes_2023 = global_df_2[global_df_2['country_norm'].duplicated(keep=False)]\
               .sort_values('country_norm')
print(dupes_2023.head(20))

# Missing counts and fractions
print(global_df_2[num_cols].isna().sum())
print(global_df_2[num_cols].isna().mean())

Empty DataFrame
Columns: [rank, country, crimeindex, pop2023, country_norm]
Index: []
crimeindex    0
pop2023       0
dtype: int64
crimeindex    0.0
pop2023       0.0
dtype: float64


In [36]:
global_df_2 = global_df_2.rename(columns={
    'crimeindex': 'crime_index_2023',
    'pop2023': 'population_2023',
})

out_path_2023 = "../data/country_safety/country_crime_pop_2023_clean.csv"
global_df_2.to_csv(out_path_2023, index=False)

In [40]:
pd.read_csv("../data/country_safety/country_crime_pop_2023_clean.csv").head()

,rank,country,crime_index_2023,population_2023,country_norm
0,1,Venezuela,83.76,28838499.0,venezuela
1,2,Papua New Guinea,80.79,10329931.0,papua new guinea
2,3,South Africa,76.86,60414495.0,south africa
3,4,Afghanistan,76.31,42239854.0,afghanistan
4,5,Honduras,74.54,10593798.0,honduras


# Quality of life dataset with saftey index - there are a bunch of these to look into that are very crediable adn complete

In [47]:
#https://www.kaggle.com/datasets/ahmedmohamed2003/quality-of-life-for-each-country
qol_1 = pd.read_csv("../data/Quality_of_Life_1.csv")
qol_1.head()

,country,Purchasing Power Value,Purchasing Power Category,Safety Value,Safety Category,Health Care Value,Health Care Category,Climate Value,Climate Category,Cost of Living Value,Cost of Living Category,Property Price to Income Value,Property Price to Income Category,Traffic Commute Time Value,Traffic Commute Time Category,Pollution Value,Pollution Category,Quality of Life Value,Quality of Life Category
0,Afghanistan,32.15,'Very Low',25.33,'Low',24.24,'Low',0.00,None,21.08,'Very Low',7.8,'Low',56.17,'Very High',84.44,'Very High',0.0,None
1,Aland Islands,125.01,'Very High',71.81,'High',79.72,'High',0.00,None,53.44,'Low',5.33,'Low',19.05,'Very Low',18.05,'Very Low',0.0,None
2,Albania,42.82,'Low',55.52,'Moderate',48.21,'Moderate',86.43,'Very High',40.85,'Low',14.88,'High',36.74,'Moderate',77.25,'High',': 104.16','Low'
3,Alderney,0.00,None,83.79,'Very High',100.00,'Very High',0.00,None,0.00,None,0.0,None,5.00,'Very Low',1.72,'Very Low',0.0,None
4,Algeria,27.60,'Very Low',47.54,'Moderate',54.43,'Moderate',94.82,'Very High',25.31,'Very Low',21.7,'Very High',45.09,'High',63.87,'High',': 98.83','Very Low'


In [48]:
qol_1.columns

Index(['country', 'Purchasing Power Value', 'Purchasing Power Category',
       'Safety Value', 'Safety Category', 'Health Care Value',
       'Health Care Category', 'Climate Value', 'Climate Category',
       'Cost of Living Value', 'Cost of Living Category',
       'Property Price to Income Value', 'Property Price to Income Category',
       'Traffic Commute Time Value', 'Traffic Commute Time Category',
       'Pollution Value', 'Pollution Category', 'Quality of Life Value',
       'Quality of Life Category'],
      dtype='object')

In [49]:
qol_1.columns = (
    qol_1.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

qol_1.columns

Index(['country', 'purchasing_power_value', 'purchasing_power_category',
       'safety_value', 'safety_category', 'health_care_value',
       'health_care_category', 'climate_value', 'climate_category',
       'cost_of_living_value', 'cost_of_living_category',
       'property_price_to_income_value', 'property_price_to_income_category',
       'traffic_commute_time_value', 'traffic_commute_time_category',
       'pollution_value', 'pollution_category', 'quality_of_life_value',
       'quality_of_life_category'],
      dtype='object')

In [50]:
qol_1['country'] = qol_1['country'].astype(str).str.strip()
qol_1['country_norm'] = qol_1['country'].str.lower()

In [52]:
#ensure numeric
value_cols = [
    'purchasing_power_value',
    'safety_value',
    'health_care_value',
    'climate_value',
    'cost_of_living_value',
    'property_price_to_income_value',
    'traffic_commute_time_value',
    'pollution_value',
    'quality_of_life_value',
]

for c in value_cols:
    qol_1[c] = pd.to_numeric(qol_1[c], errors='coerce')

In [53]:
# Duplicated country entries
dupes_qol = qol_1[qol_1['country_norm'].duplicated(keep=False)]\
             .sort_values('country_norm')
print(dupes_qol.head(20))

# Missing per column
print(qol_1[value_cols].isna().sum())
print(qol_1[value_cols].isna().mean())

Empty DataFrame
Columns: [country, purchasing_power_value, purchasing_power_category, safety_value, safety_category, health_care_value, health_care_category, climate_value, climate_category, cost_of_living_value, cost_of_living_category, property_price_to_income_value, property_price_to_income_category, traffic_commute_time_value, traffic_commute_time_category, pollution_value, pollution_category, quality_of_life_value, quality_of_life_category, country_norm]
Index: []
purchasing_power_value              0
safety_value                        0
health_care_value                   0
climate_value                       0
cost_of_living_value                0
property_price_to_income_value      3
traffic_commute_time_value          0
pollution_value                     0
quality_of_life_value             114
dtype: int64
purchasing_power_value            0.000000
safety_value                      0.000000
health_care_value                 0.000000
climate_value                     0.000000

In [54]:
for c in value_cols:
    median_val = qol_1[c].median()
    qol_1[c] = qol_1[c].fillna(median_val)

print(qol_1[value_cols].isna().sum())

purchasing_power_value            0
safety_value                      0
health_care_value                 0
climate_value                     0
cost_of_living_value              0
property_price_to_income_value    0
traffic_commute_time_value        0
pollution_value                   0
quality_of_life_value             0
dtype: int64


In [55]:
# Missing values across all columns
print(qol_1.isna().sum())

# Fraction missing
print(qol_1.isna().mean())

country                              0
purchasing_power_value               0
purchasing_power_category            0
safety_value                         0
safety_category                      0
health_care_value                    0
health_care_category                 0
climate_value                        0
climate_category                     0
cost_of_living_value                 0
cost_of_living_category              0
property_price_to_income_value       0
property_price_to_income_category    0
traffic_commute_time_value           0
traffic_commute_time_category        0
pollution_value                      0
pollution_category                   0
quality_of_life_value                0
quality_of_life_category             0
country_norm                         0
dtype: int64
country                              0.0
purchasing_power_value               0.0
purchasing_power_category            0.0
safety_value                         0.0
safety_category                      0.0
he

# deal with "none" and "0.0" and other problem values or sort out use or fill method before saving as clean

In [ ]:
out_path_qol = "../data/country_safety/country_qol_clean.csv"
qol_1.to_csv(out_path_qol, index=False)

In [14]:
cas2020 = pd.read_csv("../data/country_other/country_age.csv")
cas2020.head()

,Country,Age 0 to 14 Years,Age 15 to 64 Years,Age above 65 Years
0,Japan,12.90%,60.10%,27%
1,Italy,13.50%,63.50%,23%
2,Portugal,13.60%,64.90%,22%
3,Germany,13.10%,65.50%,22%
4,Finland,16.40%,62.40%,21%


In [15]:
print(cas2020.shape)
print(cas2020.columns)

(191, 4)
Index(['Country', 'Age 0 to 14 Years', 'Age 15 to 64 Years',
       'Age above 65 Years'],
      dtype='object')


In [16]:
# clean column names
cas2020.columns = (
    cas2020.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

cas2020 = cas2020.rename(columns={
    "country": "country",
    "age_0_to_14_years": "age_0_14",
    "age_15_to_64_years": "age_15_64",
    "age_above_65_years": "age_65_plus",
})

cas2020["country"] = cas2020["country"].astype(str).str.strip()
cas2020["country_norm"] = cas2020["country"].str.lower()

# inspect raw strings 
print("sample raw age_0_14 values:", cas2020["age_0_14"].head(10).tolist())

sample raw age_0_14 values: ['12.90%', '13.50%', '13.60%', '13.10%', '16.40%', '14.20%', '14.20%', '17.50%', '15.40%', '16.50%']


In [17]:
# remove %, commas, spaces; then convert to float
for c in ["age_0_14", "age_15_64", "age_65_plus"]:
    cas2020[c] = (
        cas2020[c]
        .astype(str)
        .str.strip()
        .str.replace("%", "")
        .str.replace(",", ".")   
        .str.replace(" ", "")
    )
    cas2020[c] = pd.to_numeric(cas2020[c], errors="coerce")

print(cas2020[["age_0_14", "age_15_64", "age_65_plus"]].head())
print(cas2020[["age_0_14", "age_15_64", "age_65_plus"]].describe())

   age_0_14  age_15_64  age_65_plus
0      12.9       60.1           27
1      13.5       63.5           23
2      13.6       64.9           22
3      13.1       65.5           22
4      16.4       62.4           21
         age_0_14   age_15_64  age_65_plus
count  191.000000  191.000000   191.000000
mean    27.647225   63.622618     8.780105
std     10.535807    6.584603     6.154487
min     11.500000   47.200000     1.000000
25%     17.900000   59.650000     4.000000
50%     26.700000   64.900000     6.000000
75%     36.600000   67.550000    14.000000
max     50.200000   85.000000    27.000000


In [18]:
print("age 2020 missing fractions:")
print(cas2020[["age_0_14", "age_15_64", "age_65_plus"]].isna().mean())

# drop any duplicate country_norm, if they exist keep first
cas2020 = cas2020.sort_values("country_norm").drop_duplicates("country_norm", keep="first").reset_index(drop=True)

age 2020 missing fractions:
age_0_14       0.0
age_15_64      0.0
age_65_plus    0.0
dtype: float64


In [19]:
cas2020.head()

,country,age_0_14,age_15_64,age_65_plus,country_norm
0,Afghanistan,43.2,54.2,3,afghanistan
1,Albania,17.4,68.9,13,albania
2,Algeria,29.3,64.5,6,algeria
3,Angola,46.8,50.8,2,angola
4,Antigua and Barbuda,23.9,69.2,7,antigua and barbuda


In [20]:
cas2020.to_csv("../data/country_other/country_age_clean.csv", index=False)
print("saved country_age_clean.csv", cas2020.shape)

saved country_age_clean.csv (191, 5)


In [5]:
cpd2020 = pd.read_csv("../data/country_other/country_pop_density.csv")
cpd2020.head()

,Rank,Country (or dependent territory),Area km2,Area mi2,Population,Density pop./km2,Density pop./mi2,Date,Population source
0,–,Macau,32.90,13,"6,76,100","20,550","53,224","September 30, 2019",Official quarterly estimate
1,1,Monaco,2.02,0.78,"38,300","18,960","49,106","December 31, 2018",Official estimate
2,2,Singapore,722.5,279,"57,03,600","7,894","20,445","July 1, 2019",Official estimate
3,–,Hong Kong,"1,106",427,"75,00,700","6,782","17,565","December 31, 2019",Official estimate
4,–,Gibraltar (UK),6.8,2.6,"33,701","4,956","12,836","July 1, 2019",UN projection


In [9]:
print(cpd2020.shape)
print(cpd2020.columns)

(251, 9)
Index(['Rank', 'Country (or dependent territory)', 'Area km2', 'Area mi2',
       'Population', 'Density pop./km2', 'Density pop./mi2', 'Date',
       'Population source'],
      dtype='object')


In [11]:
# clean column names
cpd2020.columns = (
    cpd2020.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# rename for clarity
cpd2020 = cpd2020.rename(columns={
    "country_(or_dependent_territory)": "country",
    "density_pop./km2": "density_per_km2",
    "density_pop./mi2": "density_per_mi2",
    "area_km2": "area_km2",
    "area_mi2": "area_mi2",
    "population": "population",
})

# normalize country name
cpd2020["country"] = cpd2020["country"].astype(str).str.strip()
cpd2020["country_norm"] = cpd2020["country"].str.lower()

# convert numeric columns
num_cols = ["area_km2", "area_mi2", "population", "density_per_km2", "density_per_mi2"]
for c in num_cols:
    if c in cpd2020.columns:
        cpd2020[c] = (
            cpd2020[c]
            .astype(str)
            .str.replace(",", "")
            .str.replace(" ", "")
        )
        cpd2020[c] = pd.to_numeric(cpd2020[c], errors="coerce")

print("pop density numeric summary:")
print(cpd2020[num_cols].describe())
print("pop density missing fractions:")
print(cpd2020[num_cols].isna().mean())

# drop duplicate country_norm if any
cpd2020 = cpd2020.sort_values("country_norm").drop_duplicates("country_norm", keep="first").reset_index(drop=True)

pop density numeric summary:
           area_km2      area_mi2    population  density_per_km2  \
count  2.510000e+02  2.510000e+02  2.510000e+02       251.000000   
mean   5.432937e+05  2.097769e+05  3.043882e+07       414.512669   
std    1.716009e+06  6.625517e+05  1.284182e+08      1897.989555   
min    4.400000e-01  1.700000e-01  5.600000e+01         0.030000   
25%    1.590200e+03  6.740000e+02  2.577380e+05        29.500000   
50%    6.456200e+04  2.492800e+04  4.420110e+06        82.000000   
75%    3.495840e+05  1.349750e+05  1.762174e+07       225.500000   
max    1.712524e+07  6.612093e+06  1.401812e+09     20550.000000   

       density_per_mi2  
count       251.000000  
mean       1073.583347  
std        4915.737542  
min           0.080000  
25%          76.500000  
50%         212.000000  
75%         584.000000  
max       53224.000000  
pop density missing fractions:
area_km2           0.0
area_mi2           0.0
population         0.0
density_per_km2    0.0
density_pe

In [21]:
cpd2020.head()

,rank,country,area_km2,area_mi2,population,density_per_km2,density_per_mi2,date,population_source,country_norm
0,–,Abkhazia,8660.0,3344.0,243206,28.0,73.0,"April 27, 2018",NaN,abkhazia
1,128,Afghanistan,645807.0,249347.0,31575018,49.0,127.0,"July 1, 2018",Official estimate,afghanistan
2,84,Albania,28703.0,11082.0,2862427,100.0,259.0,"January 1, 2019",Official annual estimate,albania
3,168,Algeria,2381741.0,919595.0,43000000,18.0,47.0,"January 1, 2019",Official Projection,algeria
4,–,American Samoa (US),197.0,76.0,57100,290.0,751.0,"July 1, 2020",Annual projection,american samoa (us)


In [22]:
cpd2020.to_csv("../data/country_other/country_pop_density_clean.csv", index=False)
print("saved country_pop_density_clean.csv", cpd2020.shape)

saved country_pop_density_clean.csv (251, 10)


# world cites dataset cleaning 

In [2]:
import pandas as pd
import numpy as np
import os

raw_path = "../data/global_data/worldcities.csv"

world = pd.read_csv(raw_path)
print(world.shape)
print(world.head())
print(world.columns.tolist())

(48059, 11)
        city city_ascii      lat       lng    country iso2 iso3   admin_name  \
0      Tokyo      Tokyo  35.6870  139.7495      Japan   JP  JPN        Tōkyō   
1    Jakarta    Jakarta  -6.1750  106.8275  Indonesia   ID  IDN      Jakarta   
2      Delhi      Delhi  28.6100   77.2300      India   IN  IND        Delhi   
3  Guangzhou  Guangzhou  23.1300  113.2600      China   CN  CHN    Guangdong   
4     Mumbai     Mumbai  19.0761   72.8775      India   IN  IND  Mahārāshtra   

   capital  population          id  
0  primary  37785000.0  1392685764  
1  primary  33756000.0  1360771077  
2    admin  32226000.0  1356872604  
3    admin  26940000.0  1156237133  
4    admin  24973000.0  1356226629  
['city', 'city_ascii', 'lat', 'lng', 'country', 'iso2', 'iso3', 'admin_name', 'capital', 'population', 'id']


In [3]:
# Only keep columns we care about
keep_cols = [
    "city",
    "lat",
    "lng",
    "country",
    "population",
    
]

world = world[keep_cols].copy()

In [4]:
# Strip and lowercase for normalized keys
world["city"] = world["city"].astype(str).str.strip()
world["country"] = world["country"].astype(str).str.strip()
world["country_norm"] = world["country"].astype(str).str.strip().str.lower()
world["city_norm"] = world["city"].astype(str).str.strip().str.lower()

In [8]:
world.head()

,city,lat,lon,country,population,country_norm,city_norm
0,Tokyo,35.6870,139.7495,Japan,37785000.0,japan,tokyo
1,Jakarta,-6.1750,106.8275,Indonesia,33756000.0,indonesia,jakarta
2,Delhi,28.6100,77.2300,India,32226000.0,india,delhi
3,Guangzhou,23.1300,113.2600,China,26940000.0,china,guangzhou
4,Mumbai,19.0761,72.8775,India,24973000.0,india,mumbai


In [7]:
# Rename and normalize
world = world.rename(columns={"lng": "lon"})

In [9]:
# Ensure numeric types
world["lat"] = pd.to_numeric(world["lat"], errors="coerce")
world["lon"] = pd.to_numeric(world["lon"], errors="coerce")
world["population"] = pd.to_numeric(world["population"], errors="coerce")

# Drop rows without coordinates (cannot be used for distance-based features)
before = len(world)
world = world.dropna(subset=["lat", "lon"])
after = len(world)
print(f"Dropped {before - after} rows with missing lat/lon")

# For population, keep zeros/NaN for now (we can still use them as "city presence")
missing_pop = world["population"].isna().sum()
print("Rows with missing population:", missing_pop)

Dropped 0 rows with missing lat/lon
Rows with missing population: 251


In [10]:
print("Clean shape:", world.shape)

Clean shape: (48059, 7)


In [11]:
world = world.dropna(subset=["population"])

In [12]:
print("Clean shape:", world.shape)

Clean shape: (47808, 7)


In [13]:
# Save
clean_path = "../data/global_data/worldcities_clean.csv"
world.to_csv(clean_path, index=False)
print("Saved cleaned worldcities to:", clean_path)

Saved cleaned worldcities to: ../data/global_data/worldcities_clean.csv


# new world data set with vast amount of features compiled from world bank

In [1]:
import pandas as pd
import numpy as np
import os

raw_path = "../data/global_data/country_data.csv"

world_1 = pd.read_csv(raw_path)
print(world_1.shape)
print(world_1.head())
print(world_1.columns.tolist())

(204, 38)
        gdp  sex_ratio  surface_area  life_expectancy_male  unemployment  \
0   20514.0      105.4      652864.0                  62.8          11.2   
1   15059.0      103.7       28748.0                  76.7          12.8   
2  173757.0      102.1     2381741.0                  75.4          11.5   
3    3238.0      102.3         468.0                   NaN           NaN   
4  105902.0       97.9     1246700.0                  57.8           6.8   

   imports  homicide_rate                                   currency iso2  \
0   8370.0            6.7         {'code': 'AFN', 'name': 'Afghani'}   AF   
1   5908.0            2.3             {'code': 'ALL', 'name': 'Lek'}   AL   
2  45140.0            1.4  {'code': 'DZD', 'name': 'Algerian Dinar'}   DZ   
3   1538.0            0.0            {'code': 'EUR', 'name': 'Euro'}   AD   
4  21340.0            4.8          {'code': 'AOA', 'name': 'Kwanza'}   AO   

   employment_services  ...  pop_growth           region  pop_density 

In [2]:
# Basic NA profile
na_counts = world_1.isna().sum().sort_values(ascending=False)
na_frac = (world_1.isna().mean().sort_values(ascending=False))

na_summary = pd.DataFrame({
    "na_count": na_counts,
    "na_frac": na_frac
})
print(na_summary)

                                    na_count   na_frac
co2_emissions                             59  0.289216
post_secondary_enrollment_male            33  0.161765
post_secondary_enrollment_female          33  0.161765
homicide_rate                             23  0.112745
refugees                                  14  0.068627
secondary_school_enrollment_male          14  0.068627
secondary_school_enrollment_female        14  0.068627
employment_industry                       11  0.053922
employment_agriculture                    11  0.053922
employment_services                       11  0.053922
tourists                                  10  0.049020
unemployment                              10  0.049020
primary_school_enrollment_female           8  0.039216
primary_school_enrollment_male             8  0.039216
infant_mortality                           8  0.039216
life_expectancy_female                     6  0.029412
life_expectancy_male                       6  0.029412
exports   

In [3]:
# Columns with low missingness (e.g., < 10%)
low_na_cols = na_summary[na_summary["na_frac"] < 0.10].index.tolist()
print("Low-NA columns (<10% missing):")
print(low_na_cols)

# Columns with moderate missingness (10–40%)
mid_na_cols = na_summary[(na_summary["na_frac"] >= 0.10) & (na_summary["na_frac"] < 0.40)].index.tolist()
print("\nMid-NA columns (10–40% missing):")
print(mid_na_cols)

# Columns with heavy missingness (>=40%)
high_na_cols = na_summary[na_summary["na_frac"] >= 0.40].index.tolist()
print("\nHigh-NA columns (>=40% missing):")
print(high_na_cols)

Low-NA columns (<10% missing):
['refugees', 'secondary_school_enrollment_male', 'secondary_school_enrollment_female', 'employment_industry', 'employment_agriculture', 'employment_services', 'tourists', 'unemployment', 'primary_school_enrollment_female', 'primary_school_enrollment_male', 'infant_mortality', 'life_expectancy_female', 'life_expectancy_male', 'exports', 'imports', 'fertility', 'forested_area', 'internet_users', 'iso2', 'gdp_per_capita', 'gdp', 'gdp_growth', 'surface_area', 'region', 'pop_density', 'urban_population_growth', 'pop_growth', 'name', 'urban_population', 'threatened_species', 'sex_ratio', 'capital', 'currency', 'population']

Mid-NA columns (10–40% missing):
['co2_emissions', 'post_secondary_enrollment_male', 'post_secondary_enrollment_female', 'homicide_rate']

High-NA columns (>=40% missing):
[]


In [4]:
target_cols = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "homicide_rate",
    "tourists",
]

# Basic check which of these actually exist
print("Existing target cols:", [c for c in target_cols if c in world_1.columns])

# For each target col, show which countries are missing it
for col in target_cols:
    if col not in world_1.columns:
        continue
    na_rows = world_1[world_1[col].isna()]
    if len(na_rows) == 0:
        continue
    print(f"\n=== Missing {col}: {len(na_rows)} rows ===")
    # Show minimal info to identify the country
    print(na_rows[["name", "iso2", "region", col]].to_string(index=False))

Existing target cols: ['gdp', 'gdp_per_capita', 'unemployment', 'life_expectancy_male', 'life_expectancy_female', 'infant_mortality', 'urban_population_growth', 'homicide_rate', 'tourists']

=== Missing gdp: 1 rows ===
                         name iso2          region  gdp
Holy See (Vatican City State)   VA Southern Europe  NaN

=== Missing gdp_per_capita: 1 rows ===
                         name iso2          region  gdp_per_capita
Holy See (Vatican City State)   VA Southern Europe             NaN

=== Missing unemployment: 10 rows ===
                           name iso2          region  unemployment
                        Andorra   AD Southern Europe           NaN
            Antigua And Barbuda   AG       Caribbean           NaN
                        Andorra   AD Southern Europe           NaN
            Antigua And Barbuda   AG       Caribbean           NaN
                       Dominica   DM       Caribbean           NaN
                        Grenada   GD       Caribbean  

In [5]:
master_path = "../data/compiled_model_ready/MR_cities_with_country_v1.csv"
master = pd.read_csv(master_path)

print(master.shape)
print(master.head())

# Unique countries used in the city-level model
master_countries = master["country"].sort_values().unique()
print("\nUnique countries in master city table (count =", len(master_countries), "):")
for c in master_countries:
    print(c)

# Create a normalized name to match world_1 'name'
master["country_norm"] = master["country"].str.strip().str.lower()
world_1["country_norm"] = world_1["name"].str.strip().str.lower()

# Quick check: which master countries don't match world_1 by normalized name
missing_in_world = (
    master[~master["country_norm"].isin(world_1["country_norm"])]
    ["country"]
    .sort_values()
    .unique()
)
print("\nMaster countries with no direct match in world_1 (by normalized name):")
for c in missing_in_world:
    print(c)

(509, 15)
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0         83.6          16.4            83.76            84.49   
1         82.0          18.0            76.86            77.49   
2         81.0          19.0            76.86            77.49   
3         80.7          19.3            80.79            81.93   
4         80.7          19.3            76.86            77.49   

   safetyindex_2020  age_0_14  age_15_64  age_65_plus  population  \
0             15.51      27.6       65.8          7.0  

In [6]:
# Inspect world_1 names to see what they look like
print(sorted(world_1["name"].unique()))

# Focus on a few key unmatched real countries
probe_names = [
    "moldova",
    "north macedonia",
    "norway",
    "puerto rico",
    "russia",
    "south korea",
    "syria",
    "taiwan",
    "tanzania",
    "venezuela",
    "vietnam",
    "hong kong (china)"
]

print("\nProbe matches in world_1 for selected names:")
for p in probe_names:
    matches = world_1[world_1["country_norm"].str.contains(p.split()[0], na=False)]
    if not matches.empty:
        print(f"\n=== Matches for '{p}' ===")
        print(matches[["name", "iso2", "region"]].to_string(index=False))

['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua And Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia, Plurinational State Of', 'Bosnia And Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Congo, The Democratic Republic Of The', 'Costa Rica', "Cote D'ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Holy See (Vatican City State)', 'Honduras', 'Hungary',

In [8]:
# Normalize base names
master["country_norm"] = master["country"].str.strip().str.lower()
world_1["name_norm"] = world_1["name"].str.strip().str.lower()

# Manual mapping from master 'country' -> world_1 'name'
country_name_map = {
    # Straightforward renames based on probe
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Taiwan": "Taiwan, China",                 # adjust if present in world_1
    "Hong Kong (China)": "Hong Kong SAR, China",  # may not exist in this file; safe to include

    # Countries that already match exactly (optional, for clarity)
    "Norway": "Norway",
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "United States": "United States",
    "United Kingdom": "United Kingdom",
}

# Apply mapping to create a join key
master["country_for_macro"] = master["country"].replace(country_name_map)
master["country_for_macro_norm"] = master["country_for_macro"].str.strip().str.lower()

# Recompute which master countries still have no match
missing_after_map = (
    master[~master["country_for_macro_norm"].isin(world_1["name_norm"])]
    ["country"]
    .sort_values()
    .unique()
)

print("Still missing after mapping (these are mostly US states etc.):")
for c in missing_after_map:
    print(c)

Still missing after mapping (these are mostly US states etc.):
AB
AK
AZ
BC
CA
CO
DC
FL
GA
HI
Hong Kong (China)
ID
IL
IN
Iran
KY
LA
MA
MD
MI
MN
MO
NC
NM
NV
NY
Norway
OH
OR
PA
Puerto Rico
Syria
TN
TX
Taiwan
UT
WA
WI


In [9]:
country_name_map = {
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Norway": "Norway",                        # explicit
    "Iran": "Iran, Islamic Republic Of",
    "Syria": "Syrian Arab Republic",
    "Taiwan": "Taiwan, China",                 # ok if present; ignored otherwise
    "Hong Kong (China)": "Hong Kong SAR, China",  # likely not present; safe no-op
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "United States": "United States",
    "United Kingdom": "United Kingdom",
}

In [10]:
master["country_for_macro"] = master["country"].replace(country_name_map)
master["country_for_macro_norm"] = master["country_for_macro"].str.strip().str.lower()

missing_after_map = (
    master[~master["country_for_macro_norm"].isin(world_1["name_norm"])]
    ["country"]
    .sort_values()
    .unique()
)

print("Still missing after mapping:")
for c in missing_after_map:
    print(c)

Still missing after mapping:
AB
AK
AZ
BC
CA
CO
DC
FL
GA
HI
Hong Kong (China)
ID
IL
IN
KY
LA
MA
MD
MI
MN
MO
NC
NM
NV
NY
Norway
OH
OR
PA
Puerto Rico
TN
TX
Taiwan
UT
WA
WI


In [11]:
missing = [
    "AB","AK","AZ","BC","CA","CO","DC","FL","GA","HI",
    "Hong Kong (China)","ID","IL","IN","KY","LA","MA","MD","MI","MN","MO",
    "NC","NM","NV","NY","Norway","OH","OR","PA","Puerto Rico","TN","TX",
    "Taiwan","UT","WA","WI"
]

state_like = {
    "AB","AK","AZ","BC","CA","CO","DC","FL","GA","HI","ID","IL","IN","KY",
    "LA","MA","MD","MI","MN","MO","NC","NM","NV","NY","OH","OR","PA","TN",
    "TX","UT","WA","WI"
}

actual_countries = sorted([x for x in missing if x not in state_like])
print("Actual countries/territories still missing:", actual_countries)

Actual countries/territories still missing: ['Hong Kong (China)', 'Norway', 'Puerto Rico', 'Taiwan']


In [12]:
targets = ["Hong Kong (China)", "Norway", "Puerto Rico", "Taiwan"]

print("=== Existing rows in world_1 for these names (if any) ===")
for t in targets:
    mask = world_1["name"].str.strip().str.lower() == t.strip().lower()
    rows = world_1[mask]
    print(f"\n--- {t} ---")
    if rows.empty:
        print("No exact row in world_1")
    else:
        print(rows.to_string(index=False))

=== Existing rows in world_1 for these names (if any) ===

--- Hong Kong (China) ---
No exact row in world_1

--- Norway ---
No exact row in world_1

--- Puerto Rico ---
No exact row in world_1

--- Taiwan ---
No exact row in world_1


In [13]:
# Helper: find a template row by exact name (case-insensitive)
def get_template_row(country_name):
    mask = world_1["name"].str.strip().str.lower() == country_name.strip().lower()
    if not mask.any():
        print(f"Template country '{country_name}' not found in world_1")
        return None
    return world_1[mask].iloc[0].copy()

new_rows = []

#  Norway: we know it's missing here, but we can approximate by copying from a similar Nordic country.
# Use Sweden as a proxy (or Finland if you prefer), so Norway has plausible macro values.
tpl_sweden = get_template_row("Sweden")
if tpl_sweden is not None:
    no = tpl_sweden.copy()
    no["name"] = "Norway"
    no["iso2"] = "NO"
    # region stays whatever Sweden had
    new_rows.append(no)

# Puerto Rico: copy from United States
tpl_us = get_template_row("United States")
if tpl_us is not None:
    pr = tpl_us.copy()
    pr["name"] = "Puerto Rico"
    pr["iso2"] = "PR"
    # If you, slightly tweak e.g. gdp_per_capita, unemployment, but can leave as is for now
    new_rows.append(pr)

# Taiwan: copy from China
tpl_cn = get_template_row("China")
if tpl_cn is not None:
    tw = tpl_cn.copy()
    tw["name"] = "Taiwan"
    tw["iso2"] = "TW"
    new_rows.append(tw)

# Hong Kong (China): copy from China as well
if tpl_cn is not None:
    hk = tpl_cn.copy()
    hk["name"] = "Hong Kong (China)"
    hk["iso2"] = "HK"
    new_rows.append(hk)

# Append the synthetic rows
if new_rows:
    world_1 = pd.concat([world_1, pd.DataFrame(new_rows)], ignore_index=True)

print("Newly added / approximated rows in world_1:")
print(
    world_1[world_1["name"].isin(
        ["Norway", "Puerto Rico", "Taiwan", "Hong Kong (China)"]
    )][["name", "iso2", "region", "gdp", "gdp_per_capita", "unemployment", "homicide_rate"]]
    .to_string(index=False)
)

Newly added / approximated rows in world_1:
             name iso2           region        gdp  gdp_per_capita  unemployment  homicide_rate
           Norway   NO  Northern Europe   556086.0         55766.8           6.7            1.1
      Puerto Rico   PR Northern America 20580223.0         62917.9           3.9            5.0
           Taiwan   TW     Eastern Asia 13608152.0          9531.9           4.4            0.5
Hong Kong (China)   HK     Eastern Asia 13608152.0          9531.9           4.4            0.5


# wasnt happy with above numbers so looked them up and applied below 

In [14]:
manual_values = {
    "Norway": {
        "unemployment": 4.0,
        "homicide_rate": 0.72,
        "gdp": 579_000_000_000,
        "gdp_per_capita": 104_000,
    },
    "Hong Kong (China)": {
        "unemployment": 3.3,
        "homicide_rate": 0.38,
        "gdp": 407_100_000_000,
        "gdp_per_capita": 44_725,  # slight rounding from 44725
    },
    "China": {
        "unemployment": 5.1,
        "homicide_rate": 0.44,
        "gdp": 18_743_800_000_000,
        "gdp_per_capita": 13_303,
    },
    "Taiwan": {
        "unemployment": 3.38,
        "homicide_rate": 0.80,
        "gdp": 793_000_000_000,
        "gdp_per_capita": 33_000,
    },
    "Puerto Rico": {
        "unemployment": 5.6,
        "homicide_rate": 15.3,
        "gdp": 154_000_000_000,
        "gdp_per_capita": 39_344,
    },
}

# Ensure Taiwan, Hong Kong, Puerto Rico rows exist (create if missing)
def ensure_row(name, iso2_guess):
    mask = world_1["name"].str.strip().str.lower() == name.strip().lower()
    if not mask.any():
        new = {
            "name": name,
            "iso2": iso2_guess,
            # initialize numerics as NaN; we'll overwrite the ones we care about
        }
        world_1.loc[len(world_1)] = {**{c: np.nan for c in world_1.columns}, **new}

ensure_row("Norway", "NO")
ensure_row("Hong Kong (China)", "HK")
ensure_row("China", "CN")
ensure_row("Taiwan", "TW")
ensure_row("Puerto Rico", "PR")

# Apply manual overrides
for cname, vals in manual_values.items():
    mask = world_1["name"].str.strip().str.lower() == cname.strip().lower()
    for col, v in vals.items():
        if col in world_1.columns:
            world_1.loc[mask, col] = v

print(
    world_1[
        world_1["name"].isin(
            ["Norway", "Hong Kong (China)", "China", "Taiwan", "Puerto Rico"]
        )
    ][["name", "iso2", "gdp", "gdp_per_capita", "unemployment", "homicide_rate"]]
    .to_string(index=False)
)

             name iso2          gdp  gdp_per_capita  unemployment  homicide_rate
            China   CN 1.874380e+13         13303.0          5.10           0.44
           Norway   NO 5.790000e+11        104000.0          4.00           0.72
      Puerto Rico   PR 1.540000e+11         39344.0          5.60          15.30
           Taiwan   TW 7.930000e+11         33000.0          3.38           0.80
Hong Kong (China)   HK 4.071000e+11         44725.0          3.30           0.38


In [15]:
na_summary_world1 = world_1.isna().sum().sort_values(ascending=False)
print(na_summary_world1)

co2_emissions                         59
post_secondary_enrollment_male        33
post_secondary_enrollment_female      33
homicide_rate                         23
refugees                              14
secondary_school_enrollment_male      14
secondary_school_enrollment_female    14
employment_agriculture                11
employment_services                   11
employment_industry                   11
tourists                              10
unemployment                          10
infant_mortality                       8
primary_school_enrollment_female       8
primary_school_enrollment_male         8
life_expectancy_female                 6
life_expectancy_male                   6
fertility                              5
exports                                5
imports                                5
forested_area                          4
internet_users                         2
gdp_per_capita                         1
gdp                                    1
gdp_growth      

In [16]:
macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

print("\nNaNs in v5 macro columns (world_1):")
print(world_1[macro_cols_v5].isna().sum())


NaNs in v5 macro columns (world_1):
gdp                         1
gdp_per_capita              1
unemployment               10
homicide_rate              23
life_expectancy_male        6
life_expectancy_female      6
infant_mortality            8
urban_population_growth     0
tourists                   10
dtype: int64


In [17]:
for col in macro_cols_v5:
    na_rows = world_1[world_1[col].isna()]
    if len(na_rows) == 0:
        continue
    print(f"\n=== {col}: {len(na_rows)} missing ===")
    print(na_rows[["name", "iso2", "region", col]].to_string(index=False))


=== gdp: 1 missing ===
                         name iso2          region  gdp
Holy See (Vatican City State)   VA Southern Europe  NaN

=== gdp_per_capita: 1 missing ===
                         name iso2          region  gdp_per_capita
Holy See (Vatican City State)   VA Southern Europe             NaN

=== unemployment: 10 missing ===
                           name iso2          region  unemployment
                        Andorra   AD Southern Europe           NaN
            Antigua And Barbuda   AG       Caribbean           NaN
                        Andorra   AD Southern Europe           NaN
            Antigua And Barbuda   AG       Caribbean           NaN
                       Dominica   DM       Caribbean           NaN
                        Grenada   GD       Caribbean           NaN
Micronesia, Federated States Of   FM      Micronesia           NaN
                         Monaco   MC  Western Europe           NaN
          Saint Kitts And Nevis   KN       Caribbean      

In [18]:
# Normalize for join
world_1["name_norm"] = world_1["name"].str.strip().str.lower()
master["country_norm"] = master["country"].str.strip().str.lower()

# Countries in world_1 that are both missing a macro value AND used in master
for col in macro_cols_v5:
    na_rows = world_1[world_1[col].isna()]
    if len(na_rows) == 0:
        continue
    used_mask = na_rows["name_norm"].isin(master["country_norm"])
    used_na = na_rows[used_mask]
    if len(used_na) == 0:
        continue
    print(f"\n=== {col}: missing for USED countries ({len(used_na)}) ===")
    print(used_na[["name", "iso2", "region", col]].to_string(index=False))


=== homicide_rate: missing for USED countries (1) ===
 name iso2          region  homicide_rate
Libya   LY Northern Africa            NaN

=== tourists: missing for USED countries (2) ===
       name iso2        region  tourists
Afghanistan   AF Southern Asia       NaN
Afghanistan   AF Southern Asia       NaN


In [19]:
# Libya: homicide rate ~3.7 per 100k (official-style estimate)
world_1.loc[world_1["name"] == "Libya", "homicide_rate"] = 3.7

# Afghanistan: ~7,000 foreign tourists (2023 level)
world_1.loc[world_1["name"] == "Afghanistan", "tourists"] = 7000.0

# v5 macro columns we care about
macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Re-check: NaNs in v5 macro columns only for countries actually used in master
world_1["name_norm"] = world_1["name"].str.strip().str.lower()
master["country_norm"] = master["country"].str.strip().str.lower()

print("\nRemaining NaNs in v5 macro columns for USED countries:")
for col in macro_cols_v5:
    na_rows = world_1[world_1[col].isna()]
    if len(na_rows) == 0:
        continue
    used_mask = na_rows["name_norm"].isin(master["country_norm"])
    used_na = na_rows[used_mask]
    if len(used_na) == 0:
        continue
    print(f"\n=== {col}: missing for USED countries ({len(used_na)}) ===")
    print(used_na[["name", "iso2", "region", col]].to_string(index=False))


Remaining NaNs in v5 macro columns for USED countries:


In [20]:
# Sanity-check key countries' tourism and GDP values
check_countries = ["France", "United States", "China", "Afghanistan"]

rows = world_1[world_1["name"].isin(check_countries)][
    ["name", "iso2", "gdp", "gdp_per_capita", "tourists"]
]

print(rows.to_string(index=False))

         name iso2          gdp  gdp_per_capita  tourists
  Afghanistan   AF 2.051400e+04           551.9    7000.0
  Afghanistan   AF 2.051400e+04           551.9    7000.0
        China   CN 1.874380e+13         13303.0   62900.0
       France   FR 2.778892e+06         41358.1   89322.0
       France   FR 2.778892e+06         41358.1   89322.0
United States   US 2.058022e+07         62917.9   79746.0


In [21]:
world_1.loc[world_1["name"] == "Afghanistan", "tourists"] = 7.0

print(
    world_1[world_1["name"].isin(["Afghanistan", "France", "United States", "China"])]
    [["name", "gdp", "gdp_per_capita", "tourists"]]
    .to_string(index=False)
)

         name          gdp  gdp_per_capita  tourists
  Afghanistan 2.051400e+04           551.9       7.0
  Afghanistan 2.051400e+04           551.9       7.0
        China 1.874380e+13         13303.0   62900.0
       France 2.778892e+06         41358.1   89322.0
       France 2.778892e+06         41358.1   89322.0
United States 2.058022e+07         62917.9   79746.0


In [22]:
# Build a compact country level macro df 
world_1_v5 = world_1[["name", "iso2", "region", "country_norm"] + macro_cols_v5].copy()

print("world_1_v5 shape:", world_1_v5.shape)
print(world_1_v5.head())

# Save out the cleaned macro feature table for v5
out_path = "../data/global_data/country_macro_v5.csv"
world_1_v5.to_csv(out_path, index=False)
print("Saved v5 macro features to:", out_path)

world_1_v5 shape: (208, 13)
          name iso2           region country_norm       gdp  gdp_per_capita  \
0  Afghanistan   AF    Southern Asia  afghanistan   20514.0           551.9   
1      Albania   AL  Southern Europe      albania   15059.0          5223.8   
2      Algeria   DZ  Northern Africa      algeria  173757.0          4114.7   
3      Andorra   AD  Southern Europe      andorra    3238.0         42051.6   
4       Angola   AO    Middle Africa       angola  105902.0          3437.3   

   unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  \
0          11.2            6.7                  62.8                    65.8   
1          12.8            2.3                  76.7                    80.1   
2          11.5            1.4                  75.4                    77.8   
3           NaN            0.0                   NaN                     NaN   
4           6.8            4.8                  57.8                    63.4   

   infant_mortal

In [23]:
macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

print("NaNs in v5 macro columns (world_1_v5):")
print(world_1_v5[macro_cols_v5].isna().sum())

NaNs in v5 macro columns (world_1_v5):
gdp                         1
gdp_per_capita              1
unemployment               10
homicide_rate              22
life_expectancy_male        6
life_expectancy_female      6
infant_mortality            8
urban_population_growth     0
tourists                    8
dtype: int64


In [25]:
base_v4 = pd.read_csv("../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv")

base_v4["country_norm"] = base_v4["country"].str.strip().str.lower()
world_1_v5["country_norm"] = world_1_v5["country_norm"].str.strip().str.lower()

for col in macro_cols_v5:
    na_rows = world_1_v5[world_1_v5[col].isna()]
    if len(na_rows) == 0:
        continue
    used_mask = na_rows["country_norm"].isin(base_v4["country_norm"])
    used_na = na_rows[used_mask]
    if len(used_na) == 0:
        continue
    print(f"\n=== {col}: NaNs for countries USED in v4 (world_1_v5) ===")
    print(used_na[["name", "iso2", "region", "country_norm", col]].to_string(index=False))

In [26]:
# Load v4 base (with worldpop + KNN)
base_path_v4 = "../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
base_v4 = pd.read_csv(base_path_v4)

print("Base v4 shape:", base_v4.shape)
print("world_1_v5 shape:", world_1_v5.shape)

# Normalize keys
base_v4["country_norm"] = base_v4["country"].str.strip().str.lower()
world_1_v5["country_norm"] = world_1_v5["country_norm"].str.strip().str.lower()

# Country name mapping: v4 country -> world_1_v5 name
country_name_map = {
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Iran": "Iran, Islamic Republic Of",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "United States": "United States",
    "United Kingdom": "United Kingdom",
    "Hong Kong (China)": "Hong Kong (China)",
    "Taiwan": "Taiwan",
    "Puerto Rico": "Puerto Rico",
    "Norway": "Norway",
}

base_v4["country_for_macro"] = base_v4["country"].replace(country_name_map)
base_v4["country_for_macro_norm"] = base_v4["country_for_macro"].str.strip().str.lower()

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Deduplicate world_1_v5 on country_norm
world_1_v5_dedup = (
    world_1_v5
    .sort_values(["country_norm"])
    .drop_duplicates(subset=["country_norm"], keep="first")
    .reset_index(drop=True)
)

print("world_1_v5_dedup shape:", world_1_v5_dedup.shape)

# Merge macro features into v4 base using in-memory world_1_v5
cities_v5 = base_v4.merge(
    world_1_v5_dedup[["country_norm"] + macro_cols_v5],
    left_on="country_for_macro_norm",
    right_on="country_norm",
    how="left",
    validate="many_to_one",
)

print("cities_v5 shape (should be 509 rows):", cities_v5.shape)
print("\nNaNs in v5 macro columns (cities_v5):")
print(cities_v5[macro_cols_v5].isna().sum())

# Sanity-check a few known countries
check = cities_v5[
    cities_v5["country"].isin(
        ["Venezuela", "Russia", "South Korea", "Norway", "Taiwan", "Hong Kong (China)", "United States", "France"]
    )
][["city", "country"] + macro_cols_v5]

print("\nSample rows with macro features attached:")
print(check.to_string(index=False))

Base v4 shape: (509, 65)
world_1_v5 shape: (208, 13)
world_1_v5_dedup shape: (192, 13)
cities_v5 shape (should be 509 rows): (509, 77)

NaNs in v5 macro columns (cities_v5):
gdp                        61
gdp_per_capita             61
unemployment               61
homicide_rate              61
life_expectancy_male       61
life_expectancy_female     61
infant_mortality           61
urban_population_growth    61
tourists                   61
dtype: int64

Sample rows with macro features attached:
              city           country        gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
           Caracas         Venezuela   208338.0          7212.2           9.4           36.7                  68.4                    76.1              25.7                      1.4     427.0
         Marseille            France  2778892.0         41358.1           8.3            1.2                  79.4  

In [27]:
na_mask = cities_v5["gdp"].isna()
na_countries = (
    cities_v5[na_mask][["country"]]
    .drop_duplicates()
    .sort_values("country")
)
print(na_countries.to_string(index=False))

          country
               AB
               AK
               AZ
               BC
               CA
               CO
               DC
               FL
               GA
               HI
Hong Kong (China)
               ID
               IL
               IN
               KY
               LA
               MA
               MD
               MI
               MN
               MO
               NC
               NM
               NV
               NY
           Norway
               OH
               OR
               PA
      Puerto Rico
               TN
               TX
           Taiwan
               UT
               WA
               WI


In [28]:
manual_values = {
    "Norway": {
        "iso2": "NO",
        "region": "Northern Europe",
        "unemployment": 4.0,
        "homicide_rate": 0.72,
        "gdp": 579000.0,          # scaled to match world_1_v5 GDP units (thousands of USD)
        "gdp_per_capita": 104000.0,
    },
    "Hong Kong (China)": {
        "iso2": "HK",
        "region": "Eastern Asia",
        "unemployment": 3.3,
        "homicide_rate": 0.38,
        "gdp": 407100.0,          # scaled to thousands
        "gdp_per_capita": 44725.0,
    },
    "Taiwan": {
        "iso2": "TW",
        "region": "Eastern Asia",
        "unemployment": 3.38,
        "homicide_rate": 0.80,
        "gdp": 793000.0,          # scaled to thousands
        "gdp_per_capita": 33000.0,
    },
    "Puerto Rico": {
        "iso2": "PR",
        "region": "Caribbean",
        "unemployment": 5.6,
        "homicide_rate": 15.3,
        "gdp": 154000.0,          # scaled to thousands
        "gdp_per_capita": 39344.0,
    },
}

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Create/update rows directly in world_1_v5
for cname, vals in manual_values.items():
    mask = world_1_v5["name"].str.strip().str.lower() == cname.strip().lower()

    if mask.any():
        idx = world_1_v5.index[mask][0]
    else:
        new_row = {col: np.nan for col in world_1_v5.columns}
        new_row["name"] = cname
        new_row["country_norm"] = cname.strip().lower()
        new_row["iso2"] = vals["iso2"]
        new_row["region"] = vals["region"]
        world_1_v5.loc[len(world_1_v5)] = new_row
        idx = world_1_v5.index[-1]

    world_1_v5.loc[idx, "name"] = cname
    world_1_v5.loc[idx, "country_norm"] = cname.strip().lower()
    world_1_v5.loc[idx, "iso2"] = vals["iso2"]
    world_1_v5.loc[idx, "region"] = vals["region"]

    for col in ["gdp", "gdp_per_capita", "unemployment", "homicide_rate"]:
        world_1_v5.loc[idx, col] = vals[col]

print("Special rows after force-fix:")
print(
    world_1_v5[
        world_1_v5["name"].isin(["Norway", "Hong Kong (China)", "Taiwan", "Puerto Rico"])
    ][["name", "country_norm", "iso2", "region"] + macro_cols_v5].to_string(index=False)
)

Special rows after force-fix:
             name      country_norm iso2          region      gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
           Norway            norway   NO Northern Europe 579000.0        104000.0          4.00           0.72                  80.8                    84.4               2.0                      1.1    7440.0
      Puerto Rico       puerto rico   PR       Caribbean 154000.0         39344.0          5.60          15.30                  76.3                    81.3               5.8                      0.9   79746.0
           Taiwan            taiwan   TW    Eastern Asia 793000.0         33000.0          3.38           0.80                  74.5                    79.0               9.9                      2.9   62900.0
Hong Kong (China) hong kong (china)   HK    Eastern Asia 407100.0         44725.0          3.30           0.38                  74

In [29]:
base_v4 = pd.read_csv("../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv")
base_v4["country_norm"] = base_v4["country"].str.strip().str.lower()

country_name_map = {
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Iran": "Iran, Islamic Republic Of",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "Hong Kong (China)": "Hong Kong (China)",
    "Taiwan": "Taiwan",
    "Puerto Rico": "Puerto Rico",
    "Norway": "Norway",
}

base_v4["country_for_macro"] = base_v4["country"].replace(country_name_map)
base_v4["country_for_macro_norm"] = base_v4["country_for_macro"].str.strip().str.lower()

print("Countries expected to match special rows:")
print(
    base_v4[base_v4["country"].isin(["Norway", "Hong Kong (China)", "Taiwan", "Puerto Rico"])]
    [["city", "country", "country_for_macro_norm"]]
    .drop_duplicates()
    .sort_values(["country", "city"])
    .to_string(index=False)
)

print("\nMatching rows present in world_1_v5:")
print(
    world_1_v5[world_1_v5["country_norm"].isin(["norway", "hong kong (china)", "taiwan", "puerto rico"])]
    [["name", "country_norm", "iso2", "region"] + macro_cols_v5]
    .sort_values("name")
    .to_string(index=False)
)

Countries expected to match special rows:
        city           country country_for_macro_norm
   Hong Kong Hong Kong (China)      hong kong (china)
      Bergen            Norway                 norway
Kristiansand            Norway                 norway
        Oslo            Norway                 norway
   Stavanger            Norway                 norway
   Trondheim            Norway                 norway
    San Juan       Puerto Rico            puerto rico
      Taipei            Taiwan                 taiwan

Matching rows present in world_1_v5:
             name      country_norm iso2          region      gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
Hong Kong (China) hong kong (china)   HK    Eastern Asia 407100.0         44725.0          3.30           0.38                  74.5                    79.0               9.9                      2.9   62900.0
           Nor

In [31]:
macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

print(
    world_1_v5[
        world_1_v5["name"].isin(["Norway", "Hong Kong (China)", "Taiwan", "Puerto Rico"])
    ][["name", "country_norm", "iso2", "region"] + macro_cols_v5]
    .sort_values("name")
    .to_string(index=False)
)

             name      country_norm iso2          region      gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
Hong Kong (China) hong kong (china)   HK    Eastern Asia 407100.0         44725.0          3.30           0.38                  74.5                    79.0               9.9                      2.9   62900.0
           Norway            norway   NO Northern Europe 579000.0        104000.0          4.00           0.72                  80.8                    84.4               2.0                      1.1    7440.0
      Puerto Rico       puerto rico   PR       Caribbean 154000.0         39344.0          5.60          15.30                  76.3                    81.3               5.8                      0.9   79746.0
           Taiwan            taiwan   TW    Eastern Asia 793000.0         33000.0          3.38           0.80                  74.5                    79.0    

In [32]:
# Reload base v4 
base_path_v4 = "../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
base_v4 = pd.read_csv(base_path_v4)

base_v4["country_norm"] = base_v4["country"].str.strip().str.lower()
world_1_v5["country_norm"] = world_1_v5["country_norm"].str.strip().str.lower()

country_name_map = {
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Iran": "Iran, Islamic Republic Of",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "United States": "United States",
    "United Kingdom": "United Kingdom",
    "Hong Kong (China)": "Hong Kong (China)",
    "Taiwan": "Taiwan",
    "Puerto Rico": "Puerto Rico",
    "Norway": "Norway",
}

base_v4["country_for_macro"] = base_v4["country"].replace(country_name_map)
base_v4["country_for_macro_norm"] = base_v4["country_for_macro"].str.strip().str.lower()

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Dedup the *current* patched world_1_v5
world_1_v5_dedup = (
    world_1_v5
    .sort_values("country_norm")
    .drop_duplicates(subset=["country_norm"], keep="first")
    .reset_index(drop=True)
)

print("world_1_v5_dedup shape:", world_1_v5_dedup.shape)

cities_v5 = base_v4.merge(
    world_1_v5_dedup[["country_norm"] + macro_cols_v5],
    left_on="country_for_macro_norm",
    right_on="country_norm",
    how="left",
    validate="many_to_one",
)

print("cities_v5 shape:", cities_v5.shape)
print("\nNaNs in v5 macro columns (cities_v5):")
print(cities_v5[macro_cols_v5].isna().sum())

# Focus check on the four special countries
special = cities_v5[cities_v5["country"].isin(
    ["Norway", "Hong Kong (China)", "Taiwan", "Puerto Rico"]
)][["city", "country"] + macro_cols_v5]

print("\nSpecial countries macro rows in NEW cities_v5:")
print(special.to_string(index=False))

world_1_v5_dedup shape: (196, 13)
cities_v5 shape: (509, 77)

NaNs in v5 macro columns (cities_v5):
gdp                        53
gdp_per_capita             53
unemployment               53
homicide_rate              53
life_expectancy_male       53
life_expectancy_female     53
infant_mortality           53
urban_population_growth    53
tourists                   53
dtype: int64

Special countries macro rows in NEW cities_v5:
        city           country      gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
    San Juan       Puerto Rico 154000.0         39344.0          5.60          15.30                  76.3                    81.3               5.8                      0.9   79746.0
Kristiansand            Norway 579000.0        104000.0          4.00           0.72                  80.8                    84.4               2.0                      1.1    7440.0
        Oslo     

In [33]:
# Likely state/province codes (2–3 chars, all caps, excluding known real countries)
state_like = cities_v5["country"].str.fullmatch(r"[A-Z]{2,3}")
state_rows = cities_v5[state_like.fillna(False)]

print("State/province-coded rows (sample):")
print(
    state_rows[
        ["city", "country", "country_for_macro", "country_for_macro_norm"] + macro_cols_v5
    ]
    .sort_values(["country", "city"])
    .to_string(index=False)
)

State/province-coded rows (sample):
          city country country_for_macro country_for_macro_norm  gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
    Lethbridge      AB                AB                     ab  NaN             NaN           NaN            NaN                   NaN                     NaN               NaN                      NaN       NaN
     Anchorage      AK                AK                     ak  NaN             NaN           NaN            NaN                   NaN                     NaN               NaN                      NaN       NaN
       Phoenix      AZ                AZ                     az  NaN             NaN           NaN            NaN                   NaN                     NaN               NaN                      NaN       NaN
        Tucson      AZ                AZ                     az  NaN             NaN           NaN            Na

In [34]:
# Map state/province codes to parent countries
parent_map = {
    # Canada
    "AB": "Canada",
    "BC": "Canada",

    # United States
    "AK": "United States",
    "AZ": "United States",
    "CA": "United States",
    "CO": "United States",
    "DC": "United States",
    "FL": "United States",
    "GA": "United States",
    "HI": "United States",
    "ID": "United States",
    "IL": "United States",
    "IN": "United States",
    "KY": "United States",
    "LA": "United States",
    "MA": "United States",
    "MD": "United States",
    "MI": "United States",
    "MN": "United States",
    "MO": "United States",
    "NC": "United States",
    "NM": "United States",
    "NV": "United States",
    "NY": "United States",
    "OH": "United States",
    "OR": "United States",
    "PA": "United States",
    "TN": "United States",
    "TX": "United States",
    "UT": "United States",
    "WA": "United States",
    "WI": "United States",
}

# 1) Build a lookup: parent country -> its macro values (from existing rows)
parent_macros = (
    cities_v5[cities_v5["country"].isin(["United States", "Canada"])]
    .groupby("country")[macro_cols_v5]
    .first()
)

print("Parent macro values used for fill:")
print(parent_macros)

# 2) For each state/province code, copy parent macro values into NaN cells
for state_code, parent_country in parent_map.items():
    # rows where country == state_code
    mask = cities_v5["country"] == state_code
    if not mask.any():
        continue

    if parent_country not in parent_macros.index:
        print(f"Parent country '{parent_country}' not found for state '{state_code}'")
        continue

    vals = parent_macros.loc[parent_country]

    for col in macro_cols_v5:
        cities_v5.loc[mask, col] = vals[col]

Parent macro values used for fill:
                      gdp  gdp_per_capita  unemployment  homicide_rate  \
country                                                                  
Canada          1712562.0         46192.4           5.4            1.8   
United States  20580223.0         62917.9           3.9            5.0   

               life_expectancy_male  life_expectancy_female  infant_mortality  \
country                                                                         
Canada                         80.2                    84.2               4.5   
United States                  76.3                    81.3               5.8   

               urban_population_growth  tourists  
country                                           
Canada                             1.1   21134.0  
United States                      0.9   79746.0  


In [35]:
print("\nNaNs in v5 macro columns (after state fill):")
print(cities_v5[macro_cols_v5].isna().sum())

print("\nCountries with any remaining macro NaNs:")
na_countries = (
    cities_v5[cities_v5["gdp"].isna()][["country"]]
    .drop_duplicates()
    .sort_values("country")
)
print(na_countries.to_string(index=False))

print("\nSpecial countries macro rows in cities_v5 (after all fills):")
special = cities_v5[cities_v5["country"].isin(
    ["Norway", "Hong Kong (China)", "Taiwan", "Puerto Rico"]
)][["city", "country"] + macro_cols_v5]
print(special.to_string(index=False))


NaNs in v5 macro columns (after state fill):
gdp                        0
gdp_per_capita             0
unemployment               0
homicide_rate              0
life_expectancy_male       0
life_expectancy_female     0
infant_mortality           0
urban_population_growth    0
tourists                   0
dtype: int64

Countries with any remaining macro NaNs:
Empty DataFrame
Columns: [country]
Index: []

Special countries macro rows in cities_v5 (after all fills):
        city           country      gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
    San Juan       Puerto Rico 154000.0         39344.0          5.60          15.30                  76.3                    81.3               5.8                      0.9   79746.0
Kristiansand            Norway 579000.0        104000.0          4.00           0.72                  80.8                    84.4               2.0               

In [36]:
out_path_v5 = "../data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv"
cities_v5.to_csv(out_path_v5, index=False)
print("Saved final merged v5 city table to:", out_path_v5)

Saved final merged v5 city table to: ../data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv


# re load and Final check--- note the merge was done in this file becuase memory issues

In [37]:
cities_v5_check = pd.read_csv(out_path_v5)

print("\nReloaded shape:")
print(cities_v5_check.shape)

# Global NaN check on v5 macro columns
print("\nNaNs in v5 macro columns after reload:")
print(cities_v5_check[macro_cols_v5].isna().sum())

# Show any countries still missing macro values
na_countries = (
    cities_v5_check[cities_v5_check["gdp"].isna()][["country"]]
    .drop_duplicates()
    .sort_values("country")
)
print("\nCountries still missing macro values:")
print(na_countries.to_string(index=False))

# Sanity-check important countries / mapped regions
check_countries = [
    "United States",
    "Canada",
    "Norway",
    "Hong Kong (China)",
    "Taiwan",
    "Puerto Rico",
    "Venezuela",
    "Russia",
    "South Korea",
    "France",
]

check_rows = cities_v5_check[cities_v5_check["country"].isin(check_countries)][
    ["city", "country"] + macro_cols_v5
].sort_values(["country", "city"])

print("\nSanity-check sample rows:")
print(check_rows.to_string(index=False))

# Verify row count is still 509
print("\nFinal row count:", len(cities_v5_check))


Reloaded shape:
(509, 77)

NaNs in v5 macro columns after reload:
gdp                        0
gdp_per_capita             0
unemployment               0
homicide_rate              0
life_expectancy_male       0
life_expectancy_female     0
infant_mortality           0
urban_population_growth    0
tourists                   0
dtype: int64

Countries still missing macro values:
Empty DataFrame
Columns: [country]
Index: []

Sanity-check sample rows:
                                 city           country        gdp  gdp_per_capita  unemployment  homicide_rate  life_expectancy_male  life_expectancy_female  infant_mortality  urban_population_growth  tourists
                             Brampton            Canada  1712562.0         46192.4          5.40           1.80                  80.2                    84.2               4.5                      1.1   21134.0
                           Burlington            Canada  1712562.0         46192.4          5.40           1.80               

In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path
 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook data_cleaner.ipynb to pdf
[NbConvertApp] Writing 322638 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 213849 bytes to data_cleaner.pdf
